In [2]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv(
    "Dataset.csv",
    encoding="latin1",
    low_memory=False
)

# Drop missing ratings
df = df.dropna(subset=["Rating text"])

# BINARY TARGET
rating_map = {
    "Poor": 0,
    "Average": 0,
    "Good": 1,
    "Very Good": 1,
    "Excellent": 1
}
df["Rating_Binary"] = df["Rating text"].map(rating_map)
df = df.dropna(subset=["Rating_Binary"])

# Handle categorical NaNs
cat_cols = ["Cuisines", "City", "Has Online delivery"]
for col in cat_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

# Handle numeric NaNs
num_cols = ["Average Cost for two", "Price range"]
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Features & target
X = df[
    [
        "Cuisines",
        "City",
        "Average Cost for two",
        "Price range",
        "Has Online delivery"
    ]
]
y = df["Rating_Binary"]

# Categorical feature indices
cat_features = [0, 1, 4]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# CatBoost Model
model = CatBoostClassifier(
    iterations=800,
    depth=10,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Accuracy",
    class_weights=[1, 1.5],   
    random_seed=42,
    verbose=False
)

# Train
model.fit(X_train, y_train, cat_features=cat_features)

# Predict
y_pred = model.predict(X_test)

# Results
print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 76.77245104659013

Classification Report:
               precision    recall  f1-score   support

         0.0       0.78      0.79      0.78       785
         1.0       0.76      0.74      0.75       696

    accuracy                           0.77      1481
   macro avg       0.77      0.77      0.77      1481
weighted avg       0.77      0.77      0.77      1481


Confusion Matrix:
 [[620 165]
 [179 517]]


In [7]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Load dataset 
df = pd.read_csv(
    "Dataset.csv",
    encoding="latin1",
    low_memory=False
)

# 2. Drop missing target
df = df.dropna(subset=["Rating text"])

# 3. Binary target mapping
rating_map = {
    "Poor": 0,
    "Average": 0,
    "Good": 1,
    "Very Good": 1,
    "Excellent": 1
}
df["Rating_Binary"] = df["Rating text"].map(rating_map)
df = df.dropna(subset=["Rating_Binary"])

# 4. Feature selection
base_features = [
    "Cuisines",
    "City",
    "Average Cost for two",
    "Price range",
    "Has Online delivery"
]

optional_features = [
    "Votes",
    "Aggregate rating",
    "Has Table booking",
    "Online order"
]

# Keep only available columns
features = [f for f in base_features + optional_features if f in df.columns]

X = df[features].copy()
y = df["Rating_Binary"]

# 5. Handle missing values
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

for col in cat_cols:
    X.loc[:, col] = X[col].fillna("Unknown").astype(str)

for col in num_cols:
    X.loc[:, col] = X[col].fillna(X[col].median())

# 6. Categorical feature indices
cat_features = [X.columns.get_loc(col) for col in cat_cols]

# 7. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 8. CatBoost model
model = CatBoostClassifier(
    iterations=900,
    depth=10,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Accuracy",
    class_weights=[1, 1.5],
    random_seed=42,
    verbose=False
)

# 9. Train model
model.fit(X_train, y_train, cat_features=cat_features)

# 10. Evaluate
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 100.0

Classification Report:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       785
         1.0       1.00      1.00      1.00       696

    accuracy                           1.00      1481
   macro avg       1.00      1.00      1.00      1481
weighted avg       1.00      1.00      1.00      1481

